# 工具 tools
模型(Model)是Agent的大脑，负责推理分析。而工具(Tools)则是Agent的手脚，负责执行任务，与外界交互。
![Agent工作流程](https://file-run.oss-cn-beijing.aliyuncs.com/liucheng.png)

In [19]:
from dotenv import load_dotenv

load_dotenv()

True

定义带有工具的Agent基本流程如下：
- 定义工具
- 初始化模型
- 初始化Agent，绑定模型和工具

## 1. 自定义工具
### 基于tool描述工具
在LangChain中，定义工具需要用@tool装饰器，通过装饰器来定义工具名和工具的作用

In [2]:
from langchain.tools import tool

@tool('平方根', description='计算数字的平方根')
def fun1(x: float) -> float:
  return x ** 0.5

### 使用函数名和文档注释描述工具
- 工具名：就是函数名
- 工具所需参数：函数的参数列表
- 工具作用描述：默认是函数的文档注释

In [3]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
  '''计算数字的平方根'''
  return x ** 0.5

In [5]:
# 定义一个查询天气的tool
@tool
def get_weather(location: str, units: str = 'celsius', include_forecast: bool = False) -> str:
  '''
    Get current weather and optional forecast
    Args:
      location: city name or coordinates
      units: unit of degrees
      include_forecast: dose it include the weathrer forecast
  '''
  temp = 22 if units == 'celsius' else 72
  result = f'{location}当前天气是{temp}度{units[0].upper()}'
  if include_forecast:
    result += '\n 未来5天：晴天'
  return result

### 定义Pydantic Model描述参数

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class WeatherInput(BaseModel):
  location: str = Field(description='city name or coordinates')
  unit: Literal['celsius', 'fahrenheit'] = Field(
    defalut='celsius',
    description='unit of degrees'
  )
  include_forecast: bool = Field(
    default=False,
    description='dose it include the weathrer forecast'
  )


@tool(args_schema=WeatherInput)
def get_weather(location: str, unit: str, include_forecast: bool) -> str:
  '''Get current weather and optional forecast'''
  temp = 22 if units == 'celsius' else 72
  result = f'{location}当前天气是{temp}度{units[0].upper()}'
  if include_forecast:
    result += '\n 未来5天：晴天'
  return result

工具调用方式与普通函数调用方式一致：

In [14]:
square_root.invoke({'x': 256})

16.0

In [ ]:
get_weather.invoke({'location': '杭州', 'include_forecast': True})

# 测试

In [ ]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent = create_agent(
  model='deepseek-chat',
  tools=[get_weather, square_root]
)

response = agent.stream(
  {'messages': [HumanMessage('深圳接下来几天的温度是多少')]},
  stream_mode='messages'
)

for chunk,metadata in response:
  print(chunk.content, end='', flush=True)

In [ ]:
response = agent.invoke(
  {'messages': [HumanMessage('256和188的平方根分别是？')]}
)

for chunk in response['messages']:
  print(chunk.pretty_print())

# 预定义Tool
LangChain中提供了很多预定义的Tool，方便我们使用。例如：
- tavily: 就是一个用来做web搜索的工具

In [22]:
from langchain_tavily import TavilySearch

search_tool = TavilySearch(
  max_results=5,
)

# search_tool.invoke('薛定谔的猫是什么意思？用大白话说出来')

In [ ]:
agent = create_agent(
  model='deepseek-chat',
  tools= [search_tool],
  system_prompt='你是一个智能助手，你使用工具来解决用户问题。'
)

response = agent.invoke(
  {'messages': [HumanMessage('蒸蚌是什么梗？')]}
)

for msg in response['messages']:
  msg.pretty_print()

## 优化
1是官方默认的tavily工具过于复杂，2是结果中不包含网页数据源，可信度低。

解决思路：
- 自定义tavily工具封装起来
- 结构化输出
### 自定义tavily工具
LangChain官方提供的tavily工具包含了完整的参数列表，会导致额外的流量和token消耗。因此，对于简单的业务，建议封装tavily工具使用

In [26]:
tavily = TavilySearch(
  max_result=5,
  topic='general'
)

@tool
def web_search(query: str):
  '''用于搜索网页结果'''
  return tavily.invoke(query)

### 定义结构化输出实体

In [29]:
from pydantic import BaseModel, Field

class Reference(BaseModel):
  title: str = Field(description='The title of the web page cited in the answer')
  url: str = Field(description='The url of the web page cited in the answer')

class AnswerInfo(BaseModel):
  answer: str = Field(description='The final answer for user')
  reference: list[Reference] = Field(description='The web pages cited in the answer')

agent = create_agent(
  model='deepseek-chat',
  tools= [web_search],
  response_format=AnswerInfo
)

res = agent.invoke({'messages': HumanMessage('蒸蚌是什么梗？')})

for msg in res['messages']:
  msg.pretty_print()

================================ Human Message =================================

蒸蚌是什么梗？
================================== Ai Message ==================================
Tool Calls:
  web_search (call_00_AaEfbf0R2WaWiRm4EXbV9286)
 Call ID: call_00_AaEfbf0R2WaWiRm4EXbV9286
  Args:
    query: 蒸蚌是什么梗
================================= Tool Message =================================
Name: web_search

{"query": "蒸蚌是什么梗", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.163.com/dy/article/KOAREMH3055674JT.html", "title": "萝卜纸巾分不清的“蒸蚌小猫”，拿下首个全球代言！|顶流|宠物|大开门_网易订阅", "content": "这精准治愈了害怕犯错、追求完美的现代人心态。品牌与之合作，本质上是在购买和关联这种“积极、包容、治愈”的情绪价值。\n\n参与式共创引爆传播：\n\n“萝卜纸巾”提供了一个极低的参与门槛和极高的创作自由度。它的框架简单到可以被任何人、任何品牌、在任何场景下复制和再创作。\n\n从宠物到文旅，从学生到考公党，全民都成为了内容的创作者和传播的节点。\n\n这种UGC的狂欢，是任何精心策划的广告都难以比拟的传播势能。品牌加入其中，不是生硬的广告植入，而是成为“共创者”的一员，自然不生硬。\n\n“动物营销”的信任优势：\n\n相比于人类明星可能出现的“人设塌房”风险，动物IP的形象更加纯粹、稳定且安全。\n\n它们的“人设”几乎不会崩塌，且天生具备亲近感和治愈力，能有效降低消费者对广告的抵触心理，提升品牌好感度。德祐结合公益的举措，更是将这种好感度转化为品牌美